In [8]:
name = "local"
config_dir = "/Users/sophiali/Desktop/MethylVAE/configs/train/test"
latent_dim = 128
encoder_dims = [2048, 512, 128]
input_dropout = 0.1
seed = 42
verbose = True

In [10]:
from pathlib import Path
import sys

project_root = Path.cwd().parent
sys.path.append(str(project_root / "src"))

import argparse
from datetime import datetime

from methylvae.utils.config import load_config, merge_configs_with_search_space
from methylvae.training.train import train

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
config = merge_configs_with_search_space(
    load_config(f"{config_dir}/base.yaml"),
    load_config(f"{config_dir}/data.yaml"),
    load_config(f"{config_dir}/base.yaml"),
    load_config(f"{config_dir}/train.yaml"),
    search_space=load_config(f"{config_dir}/search_space.yaml")
)

In [19]:
config['paths.experiment_dir']

'/cluster/home/t144807uhn/experiments/'

train()

In [ ]:
import lightning.pytorch as pl

from pathlib import Path
from typing import Dict

from methylvae.data.datamodule import MethylDataModule
from methylvae.models.betaVAE import BetaVAE
from methylvae.training.callbacks import configure_callbacks
from methylvae.logging.logger import configure_loggers
from methylvae.utils.config import resolve_path
from methylvae.constants import BETAVAE_CHECKPOINT_DIR

trial = None
study_name = None

%load_ext autoreload
%autoreload 2

In [ ]:
pl.seed_everything(seed, workers=True)

In [ ]:
encoder_dims = config["encoder_dims"]
model = BetaVAE(
    input_dim     = config["input_dim"],
    latent_dim    = config["latent_dim"],
    encoder_dims  = encoder_dims,
    decoder_dims  = list(reversed(encoder_dims[:-1])),
    beta          = config["beta"],
    free_bits     = config.get("free_bits", 0.5),
    input_dropout = config["input_dropout"],
    num_cycles    = config["num_cycles"],
    lr            = config["lr"],
)

In [ ]:
datamodule = MethylDataModule(
    train_adata_path = config["train_adata_path"],
    val_adata_path   = config["val_adata_path"],
    test_adata_path  = config["test_adata_path"],
    batch_size       = config["batch_size"],
    num_workers      = config["num_workers"],
)

In [ ]:
checkpoint_dir = resolve_path(
    config.get("checkpoint_dir", ""), BETAVAE_CHECKPOINT_DIR
)

if trial is not None and study_name is not None:
    checkpoint_dir = Path(checkpoint_dir) / study_name / f"trial_{trial.number}"
else:
    checkpoint_dir = Path(checkpoint_dir) / run_name

checkpoint_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
callbacks = configure_callbacks(
    trial                    = trial,
    checkpoint_dir           = str(checkpoint_dir),
    early_stopping_patience = int(config.get("early_stopping_patience", 20)),
    early_stopping_min_delta = float(config.get("early_stopping_min_delta", 1e-5)),
)
logger = configure_loggers(trial=trial, study_name=study_name or run_name)[0]

In [ ]:
trainer = pl.Trainer(
    max_epochs        = config["max_epochs"],
    callbacks         = callbacks,
    logger            = logger,
    accelerator       = "auto",
    devices           = 1,
    num_nodes         = 1,
    strategy          = "auto",
    deterministic     = False,
    log_every_n_steps = 1,
    gradient_clip_val = config.get("gradient_clip_val", 1.0),
)


In [ ]:
trainer.fit(model, datamodule = datamodule)